# circuit_train

> Training loop (run through `runTeam`), measures and frozen-weight probes for the circuit model.

In [ ]:
#| default_exp circuit_train

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import copy
from dataclasses import dataclass, fields
import numpy as np
from explRL.circuit import grid, jumpProb
from explRL.members import CircuitMember
from explRL.multi import runTeam

In [ ]:
#| export
@dataclass
class CircuitHistory:
    "Per-iteration record; headings are grid indices, angles radians, times seconds."
    t: np.ndarray          # start time of the iteration
    a: np.ndarray          # arena heading after the saccade and jump (where the fixation happens)
    c: np.ndarray          # compass heading after the saccade and jump
    direction: np.ndarray  # saccade direction, +1 CW, -1 CCW
    steps: np.ndarray      # saccade grid steps
    jumped: np.ndarray     # bump jumped after the saccade
    valence: np.ndarray    # R at a
    fixDur: np.ndarray     # fixation length
    thetaG: np.ndarray     # goal heading after this iteration's goal updates (compass)
    strength: np.ndarray   # goal strength after the updates
    mirror: np.ndarray     # (a + c) mod N after the jump

_DTYPES = (float, np.int64, np.int64, np.int64, np.int64, bool, np.int64, float, float, float, np.int64)

def _history(rows):
    cols = list(zip(*rows)) if rows else [()] * len(_DTYPES)
    return CircuitHistory(*(np.asarray(col, dtype=d) for col, d in zip(cols, _DTYPES)))

def concatHistory(parts) -> CircuitHistory:
    "Join histories run one after another (times must already continue, see `train_circuit` t0)."
    return CircuitHistory(*(np.concatenate([getattr(h, f.name) for h in parts]) for f in fields(CircuitHistory)))

def endTime(history, tS) -> float:
    "Time at which the last iteration ends (nan for an empty history)."
    return float(history.t[-1] + tS + history.fixDur[-1]) if len(history.t) else float('nan')

In [ ]:
#| export
def historyFromRecords(records) -> CircuitHistory:
    "Build a `CircuitHistory` from `TeamHistory.forAgent` records (same layout as `train_circuit`)."
    return _history([(r.t,) + r.extras for r in records])

def train_circuit(agent, env, Ttot=None, t0=0.0) -> CircuitHistory:
    "Run iterations (saccade, jump, fixation; Algorithm 3) from t0 until t0 + Ttot (default params.Ttot), as a one-member team."
    T = agent.params.Ttot if Ttot is None else float(Ttot)
    h = runTeam([CircuitMember(agent)], env, T, t0=float(t0))
    return historyFromRecords(h.records)

In [ ]:
#| export
def _wrap(x): return (x + np.pi) % (2 * np.pi) - np.pi

def goalArena(history, N) -> np.ndarray:
    "Goal heading in arena coordinates per iteration (mirror sum minus thetaG), radians in [-pi, pi)."
    return _wrap(grid(N)[history.mirror] - history.thetaG)

def goalSafeDistance(history, env) -> np.ndarray:
    "Degrees from the goal in the arena to the nearest safe-zone center, per iteration."
    return np.array([env.safeDistance(g) for g in goalArena(history, env.N)])

def heldValue(history, values, start, times, tS) -> np.ndarray:
    "Value in effect at each time: start until the first iteration ends, then the latest iteration's value."
    tEnd = history.t + history.fixDur + tS
    idx = np.searchsorted(tEnd, np.asarray(times, dtype=float), side='right') - 1
    return np.where(idx < 0, start, np.asarray(values)[np.clip(idx, 0, None)])

def piScore(history) -> float:
    "PI score: (safe - danger) fixation time over total fixation time."
    tot = history.fixDur.sum()
    return float((history.valence * history.fixDur).sum() / tot) if tot > 0 else 0.0

In [ ]:
#| export
def readout(history, N) -> dict:
    "Mean fixation and turn bias per bin of arena heading minus goal in the arena (N bins, degrees)."
    rel = np.rad2deg(_wrap(grid(N)[history.a] - goalArena(history, N)))
    edges = np.linspace(-180, 180, N + 1)
    idx = np.clip(np.digitize(rel, edges) - 1, 0, N - 1)
    fix, bias = np.full(N, np.nan), np.full(N, np.nan)
    count = np.bincount(idx, minlength=N)
    nxt = np.append(history.direction[1:], 0)          # the saccade that starts at this heading
    hasNext = np.arange(len(idx)) < len(idx) - 1
    for k in np.flatnonzero(count):
        m = idx == k
        fix[k] = history.fixDur[m].mean()
        if (m & hasNext).any():
            bias[k] = nxt[m & hasNext].mean()
    return dict(centers=(edges[:-1] + edges[1:]) / 2, fixDur=fix, turnBias=bias, count=count)

def mapJumpProb(wC, env, mirror) -> np.ndarray:
    "pJ for every arena heading a, with compass heading mirror - a ('pJ along a map')."
    return np.array([jumpProb(wC, env.ringActivity(a), (mirror - a) % env.N) for a in range(env.N)])

def stablestCompass(wC, env, mirror) -> float:
    "Circular mean over compass headings of (1 - pJ) along the map; radians in [0, 2*pi)."
    comp = grid(env.N)[(mirror - np.arange(env.N)) % env.N]
    z = ((1 - mapJumpProb(wC, env, mirror)) * np.exp(1j * comp)).sum()
    return float(np.angle(z) % (2 * np.pi))

def probe(agent, env, T=240, seed=0) -> dict:
    "Run a frozen copy (learning off, jumps on, own seed); return its history, PI and jump rate."
    ag = copy.deepcopy(agent)
    ag.rl.learnC = ag.rl.learnG = False
    ag.rl.jumps = True
    ag.rng = np.random.default_rng(seed)
    h = train_circuit(ag, env, Ttot=T)
    return dict(history=h, PI=piScore(h), jumpRate=float(h.jumped.mean()))

In [ ]:
from math import pi
from fastcore.test import test_eq, test_close
from explRL.params import Params
from explRL.core import loadCircuitAgent
from explRL.envs import ZoneArena

p = Params()
N = p.nCirc
env = ZoneArena()
quiet = dict(learnC=False, learnG=False, jumps=False)
hist = train_circuit(loadCircuitAgent(params=p, seed=0, **quiet), env, Ttot=5.0)
n = len(hist.t)
assert n > 0
for f in fields(CircuitHistory):
    test_eq(len(getattr(hist, f.name)), n)
assert hist.t[-1] < 5.0 <= endTime(hist, p.tS)
assert np.all(np.diff(hist.t) > 0)
h2 = train_circuit(loadCircuitAgent(params=p, seed=0, **quiet), env, Ttot=5.0, t0=100.0)
test_close(h2.t[0], 100.0)
assert h2.t[-1] < 105.0 <= endTime(h2, p.tS)
both = concatHistory([hist, h2])
test_eq(len(both.t), n + len(h2.t))
test_eq(len(train_circuit(loadCircuitAgent(seed=0), env, Ttot=0.0).t), 0)

**AC-7**: one seed gives one history; the mirror sum only changes by `N/2`, and only on jumps.

In [ ]:
h0 = train_circuit(loadCircuitAgent(seed=1, **quiet), env, Ttot=30.0)
h1 = train_circuit(loadCircuitAgent(seed=1, **quiet), env, Ttot=30.0)
test_close(h0.t, h1.t); test_eq(h0.a, h1.a)
h3 = train_circuit(loadCircuitAgent(seed=2, **quiet), env, Ttot=30.0)
assert len(h3.t) != len(h0.t) or not np.allclose(h0.a, h3.a)
assert np.all(h0.mirror == h0.mirror[0])
hj = train_circuit(loadCircuitAgent(seed=5, learnC=False, learnG=False, jumps=True), env, Ttot=60.0)
assert hj.jumped.any()
d = np.diff(hj.mirror) % N
assert np.all((d == 0) | (d == N // 2))
assert np.all((d == N // 2) == hj.jumped[1:])

Measures on a hand-built history:

In [ ]:
hand = CircuitHistory(
    t=np.array([0.0, 1.0, 2.0]), a=np.array([8, 8, 24]), c=np.array([0, 0, 0]),
    direction=np.array([1, -1, 1]), steps=np.array([1, 1, 1]), jumped=np.array([False, False, False]),
    valence=np.array([1, -1, 1]), fixDur=np.array([0.5, 1.0, 0.5]),
    thetaG=np.array([0.0, 0.0, 0.0]), strength=np.array([0.3, 0.4, 0.5]), mirror=np.array([8, 8, 24]))
test_close(heldValue(hand, hand.strength, 0.2, [0.0, 0.9, 2.5], p.tS), [0.2, 0.3, 0.4])
test_close(piScore(hand), (0.5 - 1.0 + 0.5) / 2.0)
test_close(goalArena(hand, N), [pi / 2, pi / 2, -pi / 2])
test_close(goalSafeDistance(hand, env), [0.0, 0.0, 0.0])
ro = readout(hand, N)
test_eq(ro['count'].sum(), 3)
k = int(np.flatnonzero(ro['count'])[0])        # all three sit at their goal (relative heading 0)
test_eq(ro['count'][k], 3)
test_close(ro['fixDur'][k], 2.0 / 3)
test_close(ro['turnBias'][k], (-1 + 1) / 2)    # next saccades of iterations 0 and 1

`pJ` along a map, the most stable compass heading, and probes:

In [ ]:
za, zs = ZoneArena(symmetric=False), ZoneArena(symmetric=True)
wC = np.zeros((N, N)); wC[0, 0] = 1.0
mjp = mapJumpProb(wC, za, mirror=0)
test_eq(len(mjp), N)
test_close(mjp[0], jumpProb(wC, za.ringActivity(0), 0))
pja = mapJumpProb(np.random.default_rng(0).uniform(0, 1, (N, N)), zs, 3)
test_close(pja[:N // 2] + pja[N // 2:], np.ones(N // 2))

wS = np.ones((N, N)); wS[:, 27] = 0.0            # compass 27 (arena 5 on map 0) never jumps
test_close(stablestCompass(wS, za, 0), grid(N)[27])

agp = loadCircuitAgent(seed=9)
wG0, wC0, st = agp.policy.wG.copy(), agp.policy.wC.copy(), agp.rng.bit_generator.state
pr = probe(agp, env, T=20.0, seed=99)
test_eq(set(pr), {'history', 'PI', 'jumpRate'})
test_close(agp.policy.wG, wG0); test_close(agp.policy.wC, wC0)
test_eq(agp.rng.bit_generator.state, st)
assert agp.rl.learnC and agp.rl.learnG
test_close(pr['jumpRate'], pr['history'].jumped.mean())

Golden fixture (Phase 5, AC-1): a 300 s run (seed 0, symmetric scene), a second 200 s segment of the same agent from `t0 = 500`, and a 60 s probe, saved from the training loop before it moved to `runTeam`, must match exactly, field by field and dtype by dtype, including the final weights and the probe's PI and jump rate.

In [ ]:
from pathlib import Path

root = Path.cwd()
while not (root / 'pyproject.toml').exists() and root != root.parent:
    root = root.parent
fx = np.load(root / 'nbs' / 'fixtures' / 'phase5' / 'circuit.npz')
def sameAsFixture(h, prefix):
    for f in fields(h):
        x, y = getattr(h, f.name), fx[prefix + f.name]
        assert x.dtype == y.dtype and np.array_equal(x, y), prefix + f.name

zs, agf = ZoneArena(symmetric=True), loadCircuitAgent(seed=0)
sameAsFixture(train_circuit(agf, zs, Ttot=300), 'seg0_')
sameAsFixture(train_circuit(agf, zs, Ttot=200, t0=500), 'seg1_')
prf = probe(agf, zs, T=60, seed=7)
sameAsFixture(prf['history'], 'probe_')
assert np.array_equal(agf.policy.wG, fx['wG']) and np.array_equal(agf.policy.wC, fx['wC'])
test_eq((prf['PI'], prf['jumpRate']), (fx['probe_PI'], fx['probe_jumpRate']))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()